# D1 source-current acquisition — fixed three-model pilot

This notebook documents and verifies the canonical acquisition route for `pilot_target_node_distribution.csv`. It preserves the archived D1 stimulus, checkpoint cohort, current extraction, edge selection, target-node aggregation, and frames 10–49 averaging. It performs no inferential statistics and does not rebuild Table S5B.

The default path is **verification only**. Optional inference writes to a temporary file and compares it with the committed canonical CSV. The canonical file is never overwritten by that comparison.


## 1. Repository and acquisition script

Run this notebook from the repository checkout. The script resolves paths relative to itself, so the working directory may be the repository root or this notebook's directory.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "scripts/generate_d1_source_currents.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the flyvis-midd repository root.")
GENERATOR = ROOT / "scripts/generate_d1_source_currents.py"
CANONICAL_CSV = ROOT / "data/precomputed/supplementary/biological_constraints/D1/pilot/pilot_target_node_distribution.csv"
print("Repository:", ROOT)
print("Generator:", GENERATOR)
print("Canonical CSV:", CANONICAL_CSV)


## 2. Frozen configuration

- FlyVis **v1.2.0**, commit `92b3845cc426dd309a1a0e1b3890156c42e14021`
- Official best checkpoints: `flow/0000/000`, `/005`, `/010`
- Stimulus: static 3MIDD, C=48, Original and Mirror
- 12 phases: 0°–13.75° in 1.25° steps; 72 total integrations
- 50 frames, `dt=0.02`, 1 s fade-in; analysis frames 10–49 inclusive
- BoxEye: extent 15, kernel size 13; source image 382 px, inner diameter 55 px, outer diameter 382 px
- Sources: Mi1/Tm3 onto T4a–d; Tm1/Tm2/Tm4/Tm9 onto T5a–d
- Signed source-edge currents are summed per target node and averaged across frames 10–49.

The frozen expected output is 1,661,184 data rows and SHA-256 `fbb384f5e7cbfc11fdad7feae0df57c95e00f4d1dd8b9342a40946e9a5d970ca`.


## 3. External FlyVis runtime and pretrained assets

The model checkpoints are official external FlyVis assets and are not bundled in this repository. In an environment with the repository's analysis dependencies, install the pinned upstream revision and retrieve pretrained assets:

```bash
python -m pip install -e ".[analysis]"
python -m pip install "git+https://github.com/TuragaLab/flyvis.git@92b3845cc426dd309a1a0e1b3890156c42e14021"
flyvis download-pretrained
```

If the data are stored outside FlyVis' default `~/flyvis/data` location, set `FLYVIS_ROOT_DIR` to that data directory. The runtime gate checks the package version/commit and the three best checkpoints before inference. A previously recorded CPU run took about 91 seconds for 72 integrations; actual time and memory depend on hardware.


## 4. Verify the retained canonical output (no inference)

This checks the canonical output's schema, row count, and SHA-256 without loading its full contents into memory. It does not alter the file.


In [ ]:
verified = subprocess.run(
    [sys.executable, str(GENERATOR), "--verify-existing"],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
print(verified.stdout)


## 5. Optional regeneration to a temporary file

Leave `REGENERATE_TO_TEMP` false for routine reproduction. If enabled, the script runs the fixed 72 integrations to a temporary CSV, validates its schema/row count, and compares it with the saved canonical CSV. It does not replace the canonical file. The explicit `--generate-canonical-if-missing` CLI action can create a missing canonical output only if its generated bytes match the recorded SHA-256; it refuses an existing file.


In [ ]:
REGENERATE_TO_TEMP = False
if REGENERATE_TO_TEMP:
    regenerated = subprocess.run(
        [sys.executable, str(GENERATOR), "--regenerate-to-temp"],
        cwd=ROOT, check=True, capture_output=True, text=True,
    )
    print(regenerated.stdout)
else:
    print("Inference skipped. Set REGENERATE_TO_TEMP=True only when the pinned runtime and official checkpoints are available.")


## 6. Reload the saved artifact

The acquisition/reproduction boundary is explicit: downstream inspection reads the CSV back from its canonical on-disk location. No new statistical summary is computed here.


In [ ]:
import hashlib
import pandas as pd

EXPECTED_ROWS = 1_661_184
EXPECTED_COLUMNS = [
    "target_type", "source_type", "target_node_id",
    "mean_current_frames_10_49", "model_id", "phase_index", "phase_deg", "condition",
]
with CANONICAL_CSV.open("rb") as stream:
    saved_sha256 = hashlib.file_digest(stream, "sha256").hexdigest()
d1_saved = pd.read_csv(CANONICAL_CSV, dtype={"model_id": str})
assert len(d1_saved) == EXPECTED_ROWS
assert d1_saved.columns.tolist() == EXPECTED_COLUMNS
assert saved_sha256 == "fbb384f5e7cbfc11fdad7feae0df57c95e00f4d1dd8b9342a40946e9a5d970ca"
print("Reloaded saved D1 artifact:", len(d1_saved), "rows; SHA-256:", saved_sha256)
display(d1_saved.head())


## 7. Relationship to Supplementary Table S5B

This three-model node-level pilot is a retained acquisition artifact, not the source of the published official-50 D1 statistics in Table S5B. Those frozen summary values remain in `data/precomputed/supplementary/biological_constraints/tables/TableS5B_activity_and_current.csv` and are displayed by `biological_constraints_replay.ipynb`. That replay notebook reads the saved table; this acquisition notebook does not recompute D1 statistics or change Table S5B.
